In [1]:
a=3
a

3

In [ ]:
zip1 = rf"F:\data.zip"
zip2 = rf"F:\zip-data.zip"
output_zip = rf"F:\training-data.zip"

chunk_size = 1024 * 1024 * 10  # 10MB chunks

with open(output_zip, "wb") as out:
    for z in [zip1, zip2]:
        with open(z, "rb") as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                out.write(chunk)

print("Two ZIP files merged successfully.")

In [1]:
zip1 = rf"F:\data.zip"
zip2 = rf"F:\zip-data.zip"
output_zip = rf"F:\training-data.zip"

chunk_size = 1024 * 1024 * 10  # 10MB chunks

with open(output_zip, "wb") as out:
    for z in [zip1, zip2]:
        with open(z, "rb") as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                out.write(chunk)

print("Two ZIP files merged successfully.")

Two ZIP files merged successfully.


In [5]:
import zipfile
from pathlib import PurePosixPath

zip_path = r"F:\trained-data.zip"

with zipfile.ZipFile(zip_path, "r") as zf:
    # Collect only top-level folder names from zip entries
    top_level_folders = {
        PurePosixPath(name).parts[0]
        for name in zf.namelist()
        if name and not name.startswith("__MACOSX/") and "/" in name
    }

folders = sorted(top_level_folders)
print(len(folders))
for folder in folders:
    print(folder)

1
Training-Data


In [6]:
import zipfile
from pathlib import PurePosixPath

zip_path = r"F:\trained-data.zip"
target_subfolder = folder  # uses previously listed/selected subfolder

with zipfile.ZipFile(zip_path, "r") as zf:
    subfolders = {
        parts[1]
        for name in zf.namelist()
        if name and not name.startswith("__MACOSX/")
        for parts in [PurePosixPath(name).parts]
        if len(parts) >= 2
        and parts[0] == target_subfolder
        and (name.endswith("/") or len(parts) >= 3)
    }

print(len(subfolders))

1431


In [2]:
import zipfile
from pathlib import PurePosixPath

zip1 = r"F:\data.zip"
zip2 = r"F:\zip-data.zip"
output_zip = r"F:\training-data.zip"
target_root = "Zip-Data"

with zipfile.ZipFile(zip1, "r") as z1, zipfile.ZipFile(zip2, "r") as z2, zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as out_zip:
    # Keep zip1 entries as they are.
    for info in z1.infolist():
        name = info.filename
        if not name or name.startswith("__MACOSX/"):
            continue
        if info.is_dir():
            out_zip.writestr(name, b"")
        else:
            out_zip.writestr(name, z1.read(name))

    # Ensure target root exists.
    out_zip.writestr(f"{target_root}/", b"")

    # Put zip2 content inside Zip-Data/...
    for info in z2.infolist():
        name = info.filename
        if not name or name.startswith("__MACOSX/"):
            continue

        parts = PurePosixPath(name).parts
        if not parts:
            continue

        # Drop zip2 top-level folder, then append remainder under Zip-Data.
        rel_parts = parts[1:] if len(parts) > 1 else parts
        new_name = str(PurePosixPath(target_root, *rel_parts))

        if info.is_dir():
            if not new_name.endswith("/"):
                new_name += "/"
            out_zip.writestr(new_name, b"")
        else:
            out_zip.writestr(new_name, z2.read(name))

print("Merged successfully: zip2 nested folders added under Zip-Data")

Merged successfully: zip2 nested folders added under Zip-Data


In [1]:
import os
from tqdm import tqdm

input_file = r"F:\\data.zip"
output_prefix = r"F:\\data_part"
part_size = 12 * 1024 * 1024 * 1024  # first part max 12GB
read_buffer = 256 * 1024 * 1024      # 256MB streaming buffer

file_size = os.path.getsize(input_file)

# Keep only 2 parts: part1 up to part_size, part2 gets the remaining bytes.
first_part_size = min(part_size, file_size)
second_part_size = max(0, file_size - first_part_size)
part_targets = [first_part_size] + ([second_part_size] if second_part_size > 0 else [])

print(f"Total file size: {file_size / (1024**3):.2f} GB")
print(f"Expected parts: {len(part_targets)}")

with open(input_file, "rb") as f:
    for part_num, target_size in enumerate(part_targets, start=1):
        output_file = f"{output_prefix}{part_num}.zip"

        # Resume support: skip if this part already exists with correct size.
        if os.path.exists(output_file) and os.path.getsize(output_file) == target_size:
            print(f"Skipping existing {output_file}")
            f.seek(target_size, 1)
            continue

        bytes_written = 0
        with open(output_file, "wb") as out, tqdm(
            total=target_size,
            unit="B",
            unit_scale=True,
            desc=f"Writing part {part_num}",
        ) as pbar:
            while bytes_written < target_size:
                to_read = min(read_buffer, target_size - bytes_written)
                chunk = f.read(to_read)
                if not chunk:
                    break
                out.write(chunk)
                bytes_written += len(chunk)
                pbar.update(len(chunk))

        if bytes_written != target_size:
            raise IOError(
                f"Part {part_num} incomplete: expected {target_size} bytes, wrote {bytes_written} bytes"
            )

        print(f"Created {output_file}")

print("Splitting completed")

Total file size: 22.42 GB
Expected parts: 2


Writing part 1: 100%|██████████| 12.9G/12.9G [11:34<00:00, 18.6MB/s]


Created F:\\data_part1.zip


Writing part 2: 100%|██████████| 11.2G/11.2G [10:00<00:00, 18.6MB/s]

Created F:\\data_part2.zip
Splitting completed
